# Car Selling Price Prediction
## Data Exploration and Preprocessing
This notebook explores the AutoTrader car advertisement dataset to prepare for building a regression model for predicting car selling prices. We will conduct data understanding, preprocessing, and visualization to lay the foundation for modeling.

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

%config InlineBackend.figure_format = "retina"

**Load and Inspect the Dataset**

In [ ]:
# Load dataset
auto = pd.read_csv("adverts.csv")

# Display basic information
auto.info()


In [ ]:
auto.head()

### Dataset Overview
- **Rows**: 402,005
- **Columns**: 12
- Key columns include `mileage`, `price`, `fuel_type`, etc.
- Observed missing values in several columns, e.g., `mileage`, `year_of_registration`.

In [ ]:
# Check for missing values
missing_values = auto.isnull().sum()
missing_percentage = (auto.isnull().mean() * 100).round(2)
missing_data = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage': missing_percentage
}).sort_values(by='Percentage', ascending=False)

# Display missing data
print("Missing values per column:")
missing_data


### Missing Values Analysis
Below is the breakdown of missing values and their percentage:
- `year_of_registration`: ~33,000 missing values (~8%).
- `mileage`: ~127 missing values (~0.03%).
- `standard_colour`: ~5,378 missing values (~1.34%).
- Other columns like `reg_code`, `body_type`, and `fuel_type` also have missing values.

These missing values may need imputation or removal depending on their context and the importance of the feature.


In [ ]:
# Check unique values for categorical features
categorical_features = ['standard_colour', 'reg_code', 'body_type', 'fuel_type', 'vehicle_condition']

for col in categorical_features:
    print(f"Feature: {col}")
    print(f"Unique Values ({len(auto[col].unique())}):\n{auto[col].unique()}\n")


### Unique Values in Categorical Features
A summary of unique values for key categorical features:
- **`reg_code`**: Region codes, likely requiring encoding.
- **`standard_colour`**: Colors of the vehicles, with some missing or uncommon entries.
- **`standard_make` and `standard_model`**: Represent the car manufacturer and model, respectively.
- **`vehicle_condition`**: Includes values such as `NEW` and `USED`.
- **`body_type`**: Represents car types, such as `SUV`, `Hatchback`.
- **`fuel_type`**: Fuel types include `Petrol`, `Diesel`, `Hybrid`, and others.

This inspection helps identify inconsistencies or rare categories that may require special handling during preprocessing.


In [ ]:
# Get summary statistics
auto.describe()

**Distribution Analysis and Visualizations**

#### Quantitative Feature Distributions

In [ ]:
# Plotting distributions for quantitative features
quantitative_features = ['mileage', 'year_of_registration', 'price']

plt.figure(figsize=(15, 10))
for i, feature in enumerate(quantitative_features, start=1):
    plt.subplot(2, 2, i)
    sns.histplot(auto[feature], kde=True, bins=500, color='blue')
    plt.title(f'Distribution of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()



#### Categorical Feature Frequencies

In [ ]:
# Frequency distribution for a few categorical features
categorical_features = ['standard_colour', 'fuel_type', 'body_type']

plt.figure(figsize=(15, 10))
for i, feature in enumerate(categorical_features, start=1):
    plt.subplot(2, 2, i)
    auto[feature].value_counts().plot(kind='bar', color='green')
    plt.title(f'Frequency of {feature}')
    plt.xlabel(feature)
    plt.ylabel('Count')

plt.tight_layout()
plt.show()


### Observations:
#### Quantitative Features
- **Mileage**: Majority of vehicles have mileage concentrated below 100,000 miles. The presence of outliers (values close to 1 million) may require filtering or scaling.
- **Year of Registration**: Clear clustering around the years 2013–2020. Anomalous values (like 999) suggest some data cleaning is required.
- **Price**: A heavily right-skewed distribution indicates the presence of luxury/high-value vehicles that could impact model performance. Most vehicles are priced below £50,000.

#### Categorical Features
- **Standard Colour**: "Black" is most frequent, followed by "White" and "Grey".
- **Fuel Type**: "Petrol" dominates, followed by "Diesel".
- **Body Type**: "Hatchback" is the most common, with "SUV" and "Saloon" also appearing frequently.


### Step 1.2: Analysis of Predictive Power of Features

#### Numeric Features

In [ ]:
# Calculate correlation for numeric features against price
numeric_features = ['mileage', 'year_of_registration']
correlations = auto[numeric_features + ['price']].corr()['price']

# Visualize correlations with scatter plots
plt.figure(figsize=(12, 6))
for i, feature in enumerate(numeric_features, start=1):
    plt.subplot(1, 2, i)
    sns.scatterplot(x=auto[feature], y=auto['price'], alpha=0.5)
    plt.title(f'{feature} vs Price')
    plt.xlabel(feature)
    plt.ylabel('Price')

plt.tight_layout()
plt.show()




#### Observations on Numeric Features
- **Mileage**:
  - Negative correlation with `price`, indicating higher mileage reduces car value.
  - Scatter plot shows a downward trend with some noise and extreme outliers.
- **Year of Registration**:
  - Positive correlation with `price`, indicating newer cars have higher value.
  - Scatter plot shows an upward trend but highlights erroneous values (e.g., year `999`).


#### Categorical Features

In [ ]:

# Analyze relationship between categorical features and price
categorical_features = ['fuel_type', 'body_type', 'vehicle_condition']
category_price_summary = {}

for feature in categorical_features:
    category_price_summary[feature] = auto.groupby(feature)['price'].mean().sort_values(ascending=False)

# Display category-wise average prices for fuel_type
category_price_summary['fuel_type']



#### Observations on Categorical Features
- **Fuel Type**:
  - Diesel Hybrid cars have the highest average price, followed by Petrol Plug-in Hybrid and Diesel Plug-in Hybrid.
  - Natural Gas vehicles have the lowest average price, suggesting limited market demand.


## Investigating Missing Years of Registration

In [ ]:
# Count the total number of cars categorized as 'new' and 'old' in 'vehicle_condition'
total_condition_counts = auto['vehicle_condition'].value_counts()

# Count the number of missing 'year_of_registration' values for 'new' and 'old' cars
missing_condition_counts = auto[auto['year_of_registration'].isnull()]['vehicle_condition'].value_counts()

total_condition_counts, missing_condition_counts


In [ ]:

# Data for visualization
conditions = ['Used', 'New']
total_counts = [total_condition_counts['USED'], total_condition_counts['NEW']]
missing_counts = [missing_condition_counts['USED'], missing_condition_counts['NEW']]

# Plotting the data
fig, ax = plt.subplots(1, 2, figsize=(12, 6), sharey=True)

# Total cars by condition
ax[0].bar(conditions, total_counts, alpha=0.7)
ax[0].set_title("Total Cars by Condition")
ax[0].set_ylabel("Number of Cars")
ax[0].set_xlabel("Vehicle Condition")
ax[0].set_ylim(0, max(total_counts) * 1.1)

# Missing year of registration by condition
ax[1].bar(conditions, missing_counts, color='orange', alpha=0.7)
ax[1].set_title("Missing Year of Registration by Condition")
ax[1].set_xlabel("Vehicle Condition")

plt.tight_layout()
plt.show()


In [ ]:
# Find the maximum value in the 'year_of_registration' column
max_year = auto['year_of_registration'].max()
max_year

In [ ]:
auto.loc[(auto["vehicle_condition"]=="NEW") & (auto["year_of_registration"].isna()), "year_of_registration"] = max_year


In [ ]:

# Recalculate missing year_of_registration counts for both NEW and USED cars
missing_condition_counts_updated = auto[auto['year_of_registration'].isnull()]['vehicle_condition'].value_counts()

# Data for updated visualization
conditions = ['Used', 'New']
missing_counts_updated = [
    missing_condition_counts_updated.get('USED', 0),
    missing_condition_counts_updated.get('NEW', 0)
]

# Plot updated missing values
fig, ax = plt.subplots(figsize=(6, 6))

ax.bar(conditions, missing_counts_updated, color=['blue', 'orange'], alpha=0.7)
ax.set_title("Updated Missing Year of Registration by Condition")
ax.set_xlabel("Vehicle Condition")
ax.set_ylabel("Number of Missing Entries")
ax.set_ylim(0, max(missing_counts_updated) * 1.1)

plt.tight_layout()
plt.show()


In [ ]:


# Filter the data for USED and NEW cars
used_cars = auto[auto['vehicle_condition'] == 'USED']
new_cars = auto[auto['vehicle_condition'] == 'NEW']

# Plot distribution of year_of_registration for USED cars
plt.figure(figsize=(12, 6))
sns.histplot(used_cars['year_of_registration'], bins=20, kde=True, color='blue', label='USED')
plt.title('Year of Registration Distribution for USED Cars')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.legend()
plt.show()

# Plot distribution of year_of_registration for NEW cars
plt.figure(figsize=(12, 6))
sns.histplot(new_cars['year_of_registration'], bins=20, kde=True, color='green', label='NEW')
plt.title('Year of Registration Distribution for NEW Cars')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.legend()
plt.show()


In [ ]:
# Check if any entries have missing year_of_registration but valid reg_code
missing_year_valid_reg = auto[(auto['year_of_registration'].isnull()) & (auto['reg_code'].notnull())]

missing_year_valid_reg_count = missing_year_valid_reg.shape[0]

missing_year_valid_reg_count

In [ ]:
def is_alpha(val):
    try: return not val.isnumeric()
    except: return False

# There are 50 entries where this ambiguity prevents the year of registration from being inferred easily
len(auto[(auto['reg_code'].apply(is_alpha)) & (auto["year_of_registration"].isna())])
missing = auto[(auto['reg_code'].apply(is_alpha)) & (auto["year_of_registration"].isna())]
missing

In [ ]:
# Update missing years of reg based on the reg code
def reg_to_year(reg_code):
    try:
        reg_code = int(reg_code)
        if reg_code > 71 or (50 > reg_code > 20): return np.nan
        return 2000 + reg_code%50
    
    except ValueError:
        if not isinstance(reg_code, str): return np.nan
        
        letters = "ABCDEFGHJKLMNPRSTXY"
        if reg_code == "V": return (1979, 1999)
        if reg_code == "W": return (1980, 2000)
        if reg_code not in letters: return np.nan
        return (1983 + letters.find(reg_code), 1963 + letters.find(reg_code))
       

auto["reg_code_year"] = auto["reg_code"].map(reg_to_year)
missing = auto[(auto['reg_code'].apply(is_alpha)) & (auto["year_of_registration"].isna())]
       


In [ ]:
# For each ambiguous year visualise the distribution of years for vehicles of the same make
missing = missing.loc[missing["reg_code_year"].notna()]
full_missing = auto.loc[missing.index]

for i in range(len(full_missing)):
    missing_model = full_missing.iloc[i]["standard_model"]
    missing_make = full_missing.iloc[i]["standard_make"]
    years = full_missing.iloc[i]["reg_code_year"]
    used = auto.loc[(auto["standard_model"] == missing_model) & (auto["year_of_registration"].notna())]
    
    plt.figure(figsize=(17, 4), dpi=240)
    ax = sns.histplot(list(used["year_of_registration"]), kde=True)

    for year in years:
        plt.plot([year, year], [0, ax.dataLim.y1], label="Possible year from reg code")

    plt.title(f"Year of reg distribution for {missing_make} {missing_model} cars.");

    mean_year = np.array(used["year_of_registration"]).mean()
    closest_year = years[0] if abs(mean_year - years[0]) < abs(mean_year - years[1]) else years[1]
    print("Mean year", mean_year)
    print("Closest year:", closest_year)
    plt.legend()
    plt.show()

    auto.iat[full_missing.index[i], 12] = closest_year

In [ ]:
# Replace reg_code_year values with year_of_registration values where year_of_registration is not null
auto['reg_code_year'] = auto['year_of_registration'].combine_first(auto['reg_code_year'])
# Replace year_of_registration values with reg_code_year values where year_of_registration is null
auto['year_of_registration'] = auto['year_of_registration'].combine_first(auto['reg_code_year'])

auto.loc[auto["year_of_registration"].isna()]

In [ ]:
# Fill remaining missing 'year_of_registration' with the mode year for the same make

# Function to get mode or None if the mode is not available
def calculate_mode(series):
    mode = series.mode()
    return mode[0] if not mode.empty else None

# Calculate the mode for each make
mode_year_per_make = auto.groupby('standard_make')['year_of_registration'].transform(calculate_mode)

# Fill missing values in 'year_of_registration' with the mode for the same make
auto['year_of_registration'].fillna(mode_year_per_make, inplace=True)

# Confirm if there are still missing values in 'year_of_registration'
remaining_missing_years = auto['year_of_registration'].isna().sum()
remaining_missing_years

In [ ]:
# Calculate the mode of 'year_of_registration' for USED cars
used_cars_mode = auto.loc[auto['vehicle_condition'] == 'USED', 'year_of_registration'].mode()[0]

# Fill the remaining missing values with the mode for USED cars
auto['year_of_registration'].fillna(used_cars_mode, inplace=True)

# Confirm if there are still missing values in 'year_of_registration'
remaining_missing_years = auto['year_of_registration'].isna().sum()
remaining_missing_years

In [ ]:
years = auto["year_of_registration"].unique()
years.sort()
years

In [ ]:
# Filter the data for USED and NEW cars
used_cars = auto[auto['vehicle_condition'] == 'USED']
new_cars = auto[auto['vehicle_condition'] == 'NEW']

# Violin plots for year_of_registration for USED and NEW cars

plt.figure(figsize=(12, 6))
sns.violinplot(x='vehicle_condition', y='year_of_registration', data=auto, split=True, inner="quart", palette="muted")
plt.title('Year of Registration by Vehicle Condition (USED vs NEW)')
plt.xlabel('Vehicle Condition')
plt.ylabel('Year of Registration')
plt.show()


## Investigating Missing Years of Registration

### Key Findings
1. **Missing Years of Registration for NEW Cars**:
   - Initially, none of the `NEW` vehicles in the dataset had valid `year_of_registration` values.
   - Plots for `NEW` cars did not display meaningful distributions as all values were `NaN`.

2. **Handling Missing Years of Registration**:
   - For `USED` cars, missing `year_of_registration` values were inferred using `reg_code` based on UK registration rules:
     - **Numeric `reg_code`**: Follow the post-2001 system (e.g., `17` → 2017).
     - **Letter `reg_code`**: Follow the pre-2001 system (e.g., `A` → 1983).
   - For `NEW` cars, the missing `year_of_registration` was successfully imputed using the **latest valid year** found in the dataset.

3. **Current Progress**:
   - **USED Cars**: Missing `year_of_registration` values have been filled where possible using `reg_code`.
   - **NEW Cars**: Missing `year_of_registration` values have been imputed with the latest valid year.

## Next Steps
1. **Generate Updated Plots**:
   - Create meaningful distributions for `USED` and `NEW` cars after handling missing values.

In [ ]:
# Filter the data for USED and NEW cars
used_cars = auto[auto['vehicle_condition'] == 'USED']
new_cars = auto[auto['vehicle_condition'] == 'NEW']

# Plot distribution of year_of_registration for USED cars
plt.figure(figsize=(12, 6))
sns.histplot(used_cars['year_of_registration'], bins=20, kde=True, color='blue')
plt.title('Year of Registration Distribution for USED Cars (Post-Imputation)')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.show()

# Plot distribution of year_of_registration for NEW cars
plt.figure(figsize=(12, 6))
sns.histplot(new_cars['year_of_registration'], bins=20, kde=True, color='green')
plt.title('Year of Registration Distribution for NEW Cars (Post-Imputation)')
plt.xlabel('Year of Registration')
plt.ylabel('Frequency')
plt.show()

## Public Reference

In [ ]:
auto.sample(1)["public_reference"]

In [ ]:
def format_entry(entry):
    year = str(entry["year_of_registration"].values[0])[:4]
    make = entry["standard_make"].values[0]
    model = entry["standard_model"].values[0]
    mileage = int(entry["mileage"].values[0])
    price = entry["price"].values[0]
    
    return f"{year} {make} {model} with {mileage:,} miles and price £{price:,}"

In [ ]:
def format_ref(entry):
    entry = str(entry["public_reference"].values[0])
    year = entry[:4]
    month = entry[4:6]
    day = entry[6:8]
    rest = entry[8:]
    
    return year, month, day, rest

Pubic reference appears to include the year, month, day that the car was listed as well as some other numbers which are possibly a unique id.
This data can be extracted and added as new features.

---
&nbsp;

&nbsp;

## Mileage

In [ ]:
# Check the number of missing values in the 'mileage' column
missing_mileage_count = auto['mileage'].isna().sum()

missing_mileage_count

In [ ]:
# Calculate the median mileage for each group of 'vehicle_condition' and 'standard_make'
median_mileage_per_group = auto.groupby(['vehicle_condition', 'standard_make'])['mileage'].transform('median')

# Fill missing values in 'mileage' using the calculated median for the group
auto['mileage'].fillna(median_mileage_per_group, inplace=True)

# Confirm if there are still missing values in 'mileage'
remaining_missing_mileage = auto['mileage'].isna().sum()
remaining_missing_mileage


## Reg Code

In [ ]:
# Check the number of missing values in the 'reg_code' column
missing_reg_code_count = auto['reg_code'].isna().sum()

missing_reg_code_count


In [ ]:
# Calculate the mode for group based imputation
mode_reg_code_per_group = auto.groupby(['standard_make', 'year_of_registration'])['reg_code'].transform(calculate_mode)

# Fill missing values in 'reg_code' with the calculated mode
auto['reg_code'].fillna(mode_reg_code_per_group, inplace=True)

# Confirm if there are still missing values in 'reg_code'
remaining_missing_reg_code = auto['reg_code'].isna().sum()
remaining_missing_reg_code

In [ ]:
auto[auto['reg_code'].isna()].head(50)

In [ ]:
# Check which groups still have missing 'reg_code' values after imputation
missing_reg_code_groups = auto[auto['reg_code'].isna()].groupby(['standard_make', 'year_of_registration']).size()

# Display the problematic groups
missing_reg_code_groups

## Colour

In [ ]:
# Check the number of missing values in the 'standard_colour' column
missing_standard_colour_count = auto['standard_colour'].isna().sum()

missing_standard_colour_count


In [ ]:
auto["standard_colour"].value_counts()

In [ ]:
# Group-based imputation using calculate_mode for 'standard_colour'
mode_colour_per_make = auto.groupby('standard_make')['standard_colour'].transform(calculate_mode)

# Fill missing values in 'standard_colour' with the calculated mode
auto['standard_colour'].fillna(mode_colour_per_make, inplace=True)

# Confirm if there are still missing values in 'standard_colour'
remaining_missing_colour = auto['standard_colour'].isna().sum()
remaining_missing_colour

In [ ]:
# Fill the remaining missing values with a global default value or the overall mode
overall_mode_colour = calculate_mode(auto['standard_colour'])

# Fill the remaining missing values
auto['standard_colour'].fillna(overall_mode_colour, inplace=True)

# Confirm if there are still missing values in 'standard_colour'
remaining_missing_colour = auto['standard_colour'].isna().sum()
remaining_missing_colour


## Body Type


In [ ]:
# Check the number of missing values in the 'body_type' column
missing_body_type_count = auto['body_type'].isna().sum()

missing_body_type_count


In [ ]:
# Group-based imputation using calculate_mode for 'body_type' based on 'standard_make' and 'standard_model'

# Apply calculate_mode function for group-based imputation
mode_body_type_per_group = auto.groupby(['standard_make', 'standard_model'])['body_type'].transform(calculate_mode)

# Fill missing values in 'body_type' with the calculated mode
auto['body_type'].fillna(mode_body_type_per_group, inplace=True)

# Confirm if there are still missing values in 'body_type'
remaining_missing_body_type = auto['body_type'].isna().sum()
remaining_missing_body_type


In [ ]:
# Fallback: Fill remaining missing values in 'body_type' using the mode for each 'standard_make'
mode_body_type_per_make = auto.groupby('standard_make')['body_type'].transform(calculate_mode)
auto['body_type'].fillna(mode_body_type_per_make, inplace=True)

# Confirm if there are still missing values in 'body_type'
remaining_missing_body_type = auto['body_type'].isna().sum()
remaining_missing_body_type


In [ ]:
# Fallback: Fill remaining missing values in 'body_type' using the mode for the corresponding 'year_of_registration'
mode_body_type_per_year = auto.groupby('year_of_registration')['body_type'].transform(calculate_mode)
auto['body_type'].fillna(mode_body_type_per_year, inplace=True)

# Confirm if there are still missing values in 'body_type'
remaining_missing_body_type = auto['body_type'].isna().sum()
remaining_missing_body_type


## Fuel Type

In [ ]:
# Check the number of missing values in the 'fuel_type' column
missing_fuel_type_count = auto['fuel_type'].isna().sum()

missing_fuel_type_count


In [ ]:
# Group-based imputation for 'fuel_type' based on 'year_of_registration', 'standard_make', and 'body_type'
mode_fuel_type_per_group = auto.groupby(['year_of_registration', 'standard_make', 'body_type'])['fuel_type'].transform(calculate_mode)

# Fill missing values in 'fuel_type' with the calculated mode
auto['fuel_type'].fillna(mode_fuel_type_per_group, inplace=True)

# Confirm if there are still missing values in 'fuel_type'
remaining_missing_fuel_type = auto['fuel_type'].isna().sum()
remaining_missing_fuel_type


In [ ]:
mode_fuel_type_per_make_body = auto.groupby(['standard_make', 'body_type'])['fuel_type'].transform(calculate_mode)
auto['fuel_type'].fillna(mode_fuel_type_per_make_body, inplace=True)
remaining_missing_fuel_type = auto['fuel_type'].isna().sum()
remaining_missing_fuel_type


In [ ]:
auto.head()

In [ ]:

# Tailored outlier handling
# Define outlier capping function with domain knowledge for Mileage
def cap_mileage_by_condition(row):
    if row['vehicle_condition'] == 'NEW':
        return row['mileage']  # Do not cap mileage for new cars
    else:
        # Cap only for used cars
        Q1 = used_cars['mileage'].quantile(0.25)
        Q3 = used_cars['mileage'].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return min(max(row['mileage'], lower_bound), upper_bound)

# Separate new and used cars for processing
used_cars = auto[auto['vehicle_condition'] == 'USED']
auto['mileage'] = auto.apply(cap_mileage_by_condition, axis=1)

# Apply percentile-based capping for Price
lower_price = auto['price'].quantile(0.01)  # 1st percentile
upper_price = auto['price'].quantile(0.99)  # 99th percentile
auto['price'] = auto['price'].clip(lower=lower_price, upper=upper_price)

# Log transformation for target and numerical features
auto['log_price'] = np.log1p(auto['price'])  # log1p avoids issues with log(0)
auto['log_mileage'] = np.log1p(auto['mileage'])

# Define predictors and target
predictors = ['log_mileage', 'standard_colour', 'standard_make', 'standard_model',
              'vehicle_condition', 'year_of_registration', 'body_type', 'fuel_type']
target = 'log_price'

X = auto[predictors]
y = auto[target]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Update feature lists
numerical_features = ['log_mileage', 'year_of_registration']
categorical_features = ['standard_colour', 'standard_make', 'standard_model',
                        'vehicle_condition', 'body_type', 'fuel_type']

# Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# Models
knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

dt_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor())
])

lr_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Parameter grids for RandomizedSearchCV
knn_param_grid = {
    'model__n_neighbors': [3, 5, 7],
    'model__metric': ['euclidean', 'manhattan']
}

dt_param_grid = {
    'model__max_depth': [3, 5, 10],
    'model__min_samples_split': [2, 5, 10]
}

# Subset for GridSearch tuning
X_train_small, _, y_train_small, _ = train_test_split(X_train, y_train, test_size=0.8, random_state=42)

# Perform Randomized Search for kNN
random_search_knn = RandomizedSearchCV(knn_pipeline, param_distributions=knn_param_grid, n_iter=5, cv=5, scoring='r2', random_state=42, n_jobs=-1)
random_search_knn.fit(X_train_small, y_train_small)
print("kNN Best Parameters:", random_search_knn.best_params_)

# Perform Randomized Search for Decision Tree
random_search_dt = RandomizedSearchCV(dt_pipeline, param_distributions=dt_param_grid, n_iter=5, cv=5, scoring='r2', random_state=42, n_jobs=-1)
random_search_dt.fit(X_train_small, y_train_small)
print("Decision Tree Best Parameters:", random_search_dt.best_params_)

# Evaluate Linear Regression directly
lr_pipeline.fit(X_train, y_train)
lr_predictions = lr_pipeline.predict(X_test)
lr_mse = mean_squared_error(y_test, lr_predictions)
lr_r2 = r2_score(y_test, lr_predictions)
print(f"Linear Regression Test Set MSE: {lr_mse:.2f}, R²: {lr_r2:.2f}")

# Evaluate Final Models on Test Set
knn_best_model = random_search_knn.best_estimator_
dt_best_model = random_search_dt.best_estimator_

knn_predictions = knn_best_model.predict(X_test)
dt_predictions = dt_best_model.predict(X_test)

knn_mse = mean_squared_error(y_test, knn_predictions)
knn_r2 = r2_score(y_test, knn_predictions)
print(f"kNN Test Set MSE: {knn_mse:.2f}, R²: {knn_r2:.2f}")

dt_mse = mean_squared_error(y_test, dt_predictions)
dt_r2 = r2_score(y_test, dt_predictions)
print(f"Decision Tree Test Set MSE: {dt_mse:.2f}, R²: {dt_r2:.2f}")